In [6]:
!python merge_langchain_cache.py cache/langchain_merged.db cache/.langchain8.db cache/.langchain.db --langchain

[warn] Could not detect LangChain tables; will fall back to all tables.
[init] Destination DB is empty. Cloning schema from: cache/.langchain8.db
[tables] Will merge tables: full_llm_cache, full_md5_llm_cache
[source] cache/.langchain8.db
[merge] full_llm_cache: +13 rows from .langchain8.db
Traceback (most recent call last):
  File "/home/apandy/Tabular/tab_small/merge_langchain_cache.py", line 203, in <module>
    main()
  File "/home/apandy/Tabular/tab_small/merge_langchain_cache.py", line 194, in main
    merge_sources(
  File "/home/apandy/Tabular/tab_small/merge_langchain_cache.py", line 142, in merge_sources
    ins, skip = copy_table(src, dest_conn, t, verbose=verbose)
  File "/home/apandy/Tabular/tab_small/merge_langchain_cache.py", line 103, in copy_table
    dest_conn.execute(f"DETACH DATABASE {alias}")
sqlite3.OperationalError: database src is locked


In [47]:
setups = [{
    "model":"qwen3:4b",
    "prompt":"V11",
    "force_no_think": True,
},
{
    "model":"qwen3:4b",
    "prompt":"V18",
    "force_no_think": True,
},
{
    "model":"gemma3n:e4b",
    "prompt":"V11"    
},
{
    "model":"gemma3n:e4b",
    "prompt":"V18",    
},
{
    "model":"mistral",
    "prompt":"V11"    
},
{
    "model":"mistral",
    "prompt":"V18"    
},
{
    "model":"llama3.2:3b",
    "prompt":"V11"
},
{
    "model":"llama3.2:3b",
    "prompt":"V18"
},
{
    "model":"gemma3:4b",
    "prompt":"V11"    
},
{
    "model":"gemma3:4b",
    "prompt":"V18"
},
{          
    "model":"llama3.2:1b",
    "prompt":"V11"
},
{
    "model":"qwen3:4b",
    "prompt":"V11",
    "force_no_think": False,
},
{
    "model":"qwen3:4b",
    "prompt":"V18",
    "force_no_think": False,
},
]

In [46]:
%load_ext autoreload
%autoreload 2
    
import cga_utils
cga_utils.setups_summary(setups)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,Model,Prompt,Item count:,Exact Match:,Value Match:,Time:
0,qwen3:4b,V11,497,53.72,69.62,0:00:03.515346
1,qwen3:4b,V18,497,59.96,75.65,0:00:03.179878
2,gemma3n:e4b,V11,497,29.78,48.49,0:45:45.701323
3,gemma3n:e4b,V18,497,42.05,54.73,1:11:07.946870
4,mistral,V11,497,17.91,21.33,0:13:36.880840
5,mistral,V18,497,41.45,42.66,0:19:35.650964
6,llama3.2:3b,V11,497,30.58,40.44,0:00:24.162917
7,llama3.2:3b,V18,497,22.33,32.60,0:12:21.546717
8,gemma3:4b,V11,497,34.81,55.53,0:25:52.908262
9,gemma3:4b,V18,497,42.66,56.74,0:42:58.292539


In [41]:
def calc_t_stats(res1,res2, cname):
    complex_correct = res1[cname].astype(int)
    simple_correct = res2[cname].astype(int)
    
    # Paired t-test
    t_stat, p_value = stats.ttest_rel(simple_correct, complex_correct)
    
    print(f"t-statistic: {t_stat:.4f}")
    print(f"p-value: {p_value:.6f}")
    
    # Szignifikancia ellenőrzése 5%-os szinten
    if p_value < 0.05:
        print("Significant difference between complex and simple prompts.")
    else:
        print("No significant difference between complex and simple prompts.")

In [42]:
import pandas as pd
from scipy import stats

setups_df = pd.DataFrame(setups)
models = set(setups_df["model"])
for model in models:
    print(model)
    comp_df = pd.read_csv(f"res/ollama__{model.replace(':','_')}__V11.csv")
    simp_df = pd.read_csv(f"res/ollama__{model.replace(':','_')}__V18.csv")
    calc_t_stats(comp_df, simp_df, "exact_match")    
    calc_t_stats(comp_df, simp_df, "value_match")
    print('---')

gemma3n:e4b
t-statistic: 5.9896
p-value: 0.000000
Significant difference between complex and simple prompts.
t-statistic: 3.3979
p-value: 0.000734
Significant difference between complex and simple prompts.
---
gemma3:4b
t-statistic: 3.3416
p-value: 0.000896
Significant difference between complex and simple prompts.
t-statistic: 0.5937
p-value: 0.552983
No significant difference between complex and simple prompts.
---
mistral
t-statistic: 9.8976
p-value: 0.000000
Significant difference between complex and simple prompts.
t-statistic: 9.4566
p-value: 0.000000
Significant difference between complex and simple prompts.
---
llama3.2:3b
t-statistic: -3.8426
p-value: 0.000138
Significant difference between complex and simple prompts.
t-statistic: -3.6184
p-value: 0.000327
Significant difference between complex and simple prompts.
---
qwen3:4b
t-statistic: 3.3577
p-value: 0.000846
Significant difference between complex and simple prompts.
t-statistic: 3.8004
p-value: 0.000162
Significant diffe